# Import

In [2]:
import warnings
# warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", module="tqdm")

import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import shutil

from torch.utils.data import random_split

from torchvision import transforms
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from utils.StegoDataset import StegoDataset
from utils.StegoPairDataSet import StegoPairDataset
from utils.StegoPairDataLoader import StegoPairDataLoader

from utils.checkpoint import save_checkpoint

from utils.transform import (
    RandomFlip,
    RandomRot,
    ToNumpy
)

from utils.transform import (
    RandomFlipList,
    RandomRotList,
    ToNumpyList,
    ToTensorList
)

from model.YeNet import YeNet

# Make DataSet

In [3]:
image_collection_folder_path = os.path.join(
    '..',
    '..',
    '..',
    'images',
    'BOSSbase',
)

cover_image_folder_path = os.path.join(image_collection_folder_path, 'cover')
stego_image_folder_path = os.path.join(image_collection_folder_path, 'stego')

transform = transforms.Compose([
    ToNumpyList(),
    # RandomRotList(),
    # RandomFlipList(),
    ToTensorList(),  
])

dataset = StegoPairDataset(
    cover_dir=cover_image_folder_path,
    stego_dir=stego_image_folder_path,
    num_samples=1000,
    transform=transform,

    bpp=0.4,
    algorithm_name='s-uniward',
    resize_strategy='center_crop',
    W=256,
    H=256,
)

Найдено 7000 stego файлов по заданным параметрам
Загружено 1000 пар изображений


# Make DataLoader

In [4]:
# Параметры разделения
val_split = 0.2  # 20% на валидацию
batch_size = 32
random_seed = 42

# Устанавливаем seed для воспроизводимости
torch.manual_seed(random_seed)
np.random.seed(random_seed)

# Вычисляем размеры выборок
dataset_size = len(dataset)
val_size = int(val_split * dataset_size)
train_size = dataset_size - val_size

print(f"Общий размер датасета: {dataset_size}")
print(f"Тренировочная выборка: {train_size}")
print(f"Валидационная выборка: {val_size}")

# Разделяем датасет
train_dataset, val_dataset = random_split(
    dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(random_seed)
)

# Создаем DataLoader'ы для каждой выборки
train_loader = StegoPairDataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,  # перемешиваем только тренировочные данные
    num_workers=4,
    pin_memory=False,
    drop_last=False
)

val_loader = StegoPairDataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,  # не перемешиваем валидационные данные
    num_workers=4,
    pin_memory=False,
    drop_last=False
)

print(f"Тренировочных батчей: {len(train_loader)}")
print(f"Валидационных батчей: {len(val_loader)}")

Общий размер датасета: 1000
Тренировочная выборка: 800
Валидационная выборка: 200
Тренировочных батчей: 50
Валидационных батчей: 13


# Train

## No validation

In [6]:
# Параметры пакетного обучения
num_epochs = 20      # Количество эпох
log_batch_period = 10 # Период логирования в батчах
verbose = False

net = YeNet(with_bn=True)

# Устанавливаем модель в режим обучения
net.train()

criterion = nn.CrossEntropyLoss().cuda()
optimizer = optim.Adadelta(
    net.parameters(), 
    lr=0.01, 
    rho=0.95, 
    eps=1e-8,
    weight_decay=5e-4
)

# Списки для хранения метрик
epoch_losses = []
epoch_accuracies = []
batch_losses = []
batch_accuracies = []

for epoch in range(num_epochs):
    # Устанавливаем модель в режим обучения
    net.train()
    
    running_loss = 0.0
    running_accuracy = 0.0
    
    progress_bar = tqdm(train_loader, desc=f'Эпоха {epoch+1}/{num_epochs}')    
    for batch_idx, batch_data in enumerate(progress_bar):
        images = batch_data['images']
        labels = batch_data['labels']
        
        optimizer.zero_grad()
        outputs = net(images) 
        
        loss = criterion(outputs, labels)
        
        predictions = outputs.argmax(dim=1)
        accuracy = (predictions == labels).float().mean().item()
        
        # Сохраняем метрики для каждого батча
        batch_losses.append(loss.item())
        batch_accuracies.append(accuracy)
        
        # Обратный проход
        loss.backward()
        optimizer.step()
        
        # Обновляем бегущую статистику
        running_loss += loss.item()
        running_accuracy += accuracy
        
        # Обновляем прогресс-бар
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{accuracy:.4f}',
            'avg_loss': f'{running_loss/(batch_idx+1):.4f}'
        })
        
        # Печатаем подробную информацию каждые несколько батчей
        if verbose and (batch_idx + 1) % log_batch_period == 0:
            current_avg_loss = running_loss / (batch_idx + 1)
            current_avg_acc = running_accuracy / (batch_idx + 1)
            print(f'  Батч {batch_idx+1}/{len(train_loader)}: '
                  f'Loss = {loss.item():.4f}, Acc = {accuracy:.4f}, '
                  f'Avg Loss = {current_avg_loss:.4f}, Avg Acc = {current_avg_acc:.4f}')
    
    # Статистика за эпоху
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = running_accuracy / len(train_loader)
    epoch_losses.append(epoch_loss)
    epoch_accuracies.append(epoch_acc)
    
    print(f'\nЭпоха {epoch+1} завершена:')
    print(f'  Средний Loss: {epoch_loss:.4f}')
    print(f'  Средняя Accuracy: {epoch_acc:.4f}')
    print('-' * 40)

print("\nОбучение завершено!")

Эпоха 1/20:   2%|▏         | 1/50 [00:05<04:40,  5.72s/it, loss=0.7015, acc=0.5000, avg_loss=0.7015]

: 

In [ ]:
import numpy as np

# Вывод итоговых результатов
print(f"\nИтоговые результаты:")
print(f"  Лучший Loss: {min(epoch_losses):.4f} (эпоха {np.argmin(epoch_losses) + 1})")
print(f"  Лучшая Accuracy: {max(epoch_accuracies):.4f} (эпоха {np.argmax(epoch_accuracies) + 1})")
print(f"  Финальный Loss: {epoch_losses[-1]:.4f}")
print(f"  Финальная Accuracy: {epoch_accuracies[-1]:.4f}")

# Визуализация результатов
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

# График потерь по батчам
plt.subplot(1, 3, 1)
plt.plot(batch_losses, 'b-', alpha=0.5, linewidth=0.5)
plt.title('Потери по батчам')
plt.xlabel('Батч')
plt.ylabel('Loss')
plt.grid(True)

# График точности по батчам
plt.subplot(1, 3, 2)
plt.plot(batch_accuracies, 'g-', alpha=0.5, linewidth=0.5)
plt.title('Точность по батчам')
plt.xlabel('Батч')
plt.ylabel('Accuracy')
plt.grid(True)

# График по эпохам
plt.subplot(1, 3, 3)
epochs = range(1, num_epochs + 1)
plt.plot(epochs, epoch_losses, 'b-o', label='Loss')
plt.plot(epochs, epoch_accuracies, 'g-o', label='Accuracy')
plt.title('Метрики по эпохам')
plt.xlabel('Эпоха')
plt.ylabel('Значение')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## With Validation

In [5]:
import logging
import sys

# Настройка логирования: вывод в файл и в консоль
logging.basicConfig(
    level=logging.INFO,
    format='%(message)s',  # только сообщение, без лишних деталей
    handlers=[
        logging.FileHandler("training.log", mode='w'),  # перезапись файла при каждом запуске
        # logging.StreamHandler(sys.stdout)
    ]
)

In [ ]:
# Параметры early stopping
patience = 5  # количество эпох для ожидания улучшения
min_delta = 0.001  # минимальное улучшение для сброса счетчика
best_val_acc = 0.0
patience_counter = 0
early_stop = False

# Директория для сохранения чекпоинтов
checkpoint_dir = 'checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Устанавливаем seed для воспроизводимости
torch.manual_seed(random_seed)
np.random.seed(random_seed)

# Параметры пакетного обучения
num_epochs = 40      # Количество эпох
log_batch_period = 10 # Период логирования в батчах
verbose = False

# Почему то авторы
net = YeNet(with_bn=False)

# Устанавливаем модель в режим обучения
net.train()

criterion = nn.CrossEntropyLoss()

learning_rate = 1e-4
optimizer = optim.Adadelta(
    net.parameters(), 
    lr=learning_rate, 
    rho=0.95, 
    eps=1e-8,
    weight_decay=5e-4
)

from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts, StepLR

scheduler_step = StepLR(
    optimizer,
    step_size=10,          # уменьшаем каждые 15 эпох
    gamma=0.5,              # множитель уменьшения
)

# Списки для хранения метрик
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
batch_losses = []
batch_accuracies = []

# Функция валидации
def validate(model, val_loader, criterion):
    """Проверяет модель на валидационной выборке."""
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_data in val_loader:
            images = batch_data['images']
            labels = batch_data['labels']
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total
    
    return avg_val_loss, val_accuracy

for epoch in range(num_epochs):
    if early_stop:
        print(f"\n🛑 Early stopping на эпохе {epoch}")
        break
    
    print(f"\n{'='*50}")
    print(f"Эпоха {epoch+1}/{num_epochs}")
    print('='*50)
    
    running_loss = 0.0
    running_accuracy = 0.0
    
    progress_bar = tqdm(train_loader, desc=f'Обучение')    
    for batch_idx, batch_data in enumerate(progress_bar):
        net.train()
        
        images = batch_data['images']
        labels = batch_data['labels']
        
        optimizer.zero_grad()
        outputs = net(images) 
        
        loss = criterion(outputs, labels)
        
        predictions = outputs.argmax(dim=1)
        accuracy = (predictions == labels).float().mean().item()

        
        # Сохраняем метрики для каждого батча
        batch_losses.append(loss.item())
        batch_accuracies.append(accuracy)

        # Logging:
        logging.info(f"\nepoch {epoch}, batch_id {batch_idx}")
        logging.info(f"predictions: {predictions}")
        logging.info(f"labels: {labels}")
        logging.info(f"accuracy: {accuracy}")
        logging.info(f"loss: {loss.item()}")
        
        # Обратный проход
        loss.backward()
        optimizer.step()
        
        # Обновляем бегущую статистику
        running_loss += loss.item()
        running_accuracy += accuracy
        
        # Обновляем прогресс-бар
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{accuracy:.4f}',
            'avg_loss': f'{running_loss/(batch_idx+1):.4f}'
        })
    
    # Статистика за эпоху (тренировка)
    epoch_train_loss = running_loss / len(train_loader)
    epoch_train_acc = 100 * running_accuracy / len(train_loader)
    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_acc)
    
    # ===== ВАЛИДАЦИЯ =====
    epoch_val_loss, epoch_val_acc = validate(net, val_loader, criterion)
    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)
    
    # Вывод результатов эпохи
    print(f"\n📊 Результаты эпохи {epoch+1}:")
    print(f"  Тренировка - Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.2f}%")
    print(f"  Валидация   - Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.2f}%")
    
    # ===== ДЕТЕКЦИЯ ПЕРЕОБУЧЕНИЯ =====
    if epoch > 0:
        train_acc_improved = epoch_train_acc > train_accuracies[-2]
        val_acc_decreased = epoch_val_acc < val_accuracies[-2]
        
        if train_acc_improved and val_acc_decreased:
            patience_counter += 1
            print(f"  ⚠️ Признак переобучения! Счетчик: {patience_counter}/{patience}")
            
            if patience_counter >= patience:
                print(f"  🛑 Ранняя остановка: валидационная точность падает {patience} эпох подряд")
                early_stop = True
        else:
            if patience_counter > 0:
                patience_counter = max(0, patience_counter - 1)  # небольшое ослабление
    
    # ===== СОХРАНЕНИЕ ЧЕКПОИНТА =====
    is_best = epoch_val_acc > best_val_acc
    if is_best:
        best_val_acc = epoch_val_acc
        patience_counter = 0  # сбрасываем счетчик при улучшении
    
    checkpoint_state = {
        'epoch': epoch + 1,
        'state_dict': net.state_dict(),
        'optimizer': optimizer.state_dict(),
        'train_loss': epoch_train_loss,
        'train_acc': epoch_train_acc,
        'val_loss': epoch_val_loss,
        'val_acc': epoch_val_acc,
        'best_val_acc': best_val_acc,
        'patience_counter': patience_counter,
    }
    
    save_checkpoint(checkpoint_state, is_best, f'checkpoints/checkpoint_epoch_{epoch+1}.pth.tar')

print("\n" + "="*50)
print("🎉 Обучение завершено!")
print("="*50)
print(f"Лучшая точность на валидации: {best_val_acc:.2f}%")


Эпоха 1/40


Обучение:   0%|          | 0/50 [00:00<?, ?it/s]

Обучение:  52%|█████▏    | 26/50 [01:18<01:10,  2.92s/it, loss=0.6933, acc=0.5000, avg_loss=2.0251] 

: 